In [2]:

# 6_cluster.ipynb
#
# Clusters the synthetic population at national, LA, or MSOA level.
# CLUSTER_LEVEL controls the granularity — change it and re-run.
#
# Each geography unit is loaded individually from parquet (using filters),
# clustered, and its DNA rows appended to a CSV — no full dataset in memory.
#
# For each geography unit × employment group:
#   1. Fit Mx+c normalisation coefficients in memory on that group
#   2. Apply normalisation
#   3. Run KMeans
#   4. Append DNA row to CSV from raw (un-normalised) values

import sys, os, gc
sys.path.insert(0, os.path.abspath('..'))

import importlib
from data_processing.config_paths import USE_TEST_DATA, DATA_FOLDER
import data_processing.config_variables as _cv
import data_processing.config_cluster   as _cc
importlib.reload(_cv)
importlib.reload(_cc)

import pandas as pd
import numpy  as np
from pathlib import Path
from tqdm    import tqdm

import data_processing.normalise     as normalise
import data_processing.cluster_funcs as cf
importlib.reload(normalise)
importlib.reload(cf)

from data_processing.config_variables import (
    CLUSTER_VARS, SUMMARY_VARS, VARIABLE_MAP,
    CATEGORICAL_VARS, CATEGORY_MAPS,
)
from data_processing.config_cluster import WAVE, GROUPS, K_DEFAULT

# ── Config ────────────────────────────────────────────────────────────────────
CLUSTER_LEVEL    = "LA"          # "national" | "LA" | "MSOA"
SYNPOP_PARQUET   = f"../{DATA_FOLDER}/5_synthetic_population/synthetic_population.parquet"
OUTPUT_DIR       = Path(f"../{DATA_FOLDER}/6_cluster")
MIN_CLUSTER_SIZE = 5

# UNIT_FILTER — restrict which units are clustered.
#   None              → all units (default, ~350 LAs, slow)
#   "london"          → 33 London boroughs only (E09* codes)
#   ["E09000001", …]  → explicit list of codes
UNIT_FILTER = "london"

_LEVEL_COL = {
    "national": None,
    "LA":        "ladcd",
    "MSOA":      "msoa21cd",
}
if CLUSTER_LEVEL not in _LEVEL_COL:
    raise ValueError(f"CLUSTER_LEVEL must be one of {list(_LEVEL_COL)}")
LEVEL_COL = _LEVEL_COL[CLUSTER_LEVEL]

if not os.path.exists(SYNPOP_PARQUET):
    raise FileNotFoundError(f"{SYNPOP_PARQUET} not found — run 5_synthetic_population.ipynb first.")

# Derive output filename from filter so runs don't overwrite each other
_filter_tag = (
    ""         if UNIT_FILTER is None else
    "_london"  if UNIT_FILTER == "london" else
    "_custom"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / f"{CLUSTER_LEVEL}{_filter_tag}_clusters.csv"

# Always start fresh — avoids stale rows from previous runs with different columns
if OUTPUT_CSV.exists():
    OUTPUT_CSV.unlink()
    print(f"Removed existing {OUTPUT_CSV.name}")

# ── Feature columns ───────────────────────────────────────────────────────────
# Read schema only to validate feature columns without loading data
import pyarrow.parquet as pq
parquet_cols = pq.read_schema(SYNPOP_PARQUET).names
feature_cols = [f"{WAVE}_{b}" for b in CLUSTER_VARS if f"{WAVE}_{b}" in parquet_cols]
missing_feat = [f"{WAVE}_{b}" for b in CLUSTER_VARS if f"{WAVE}_{b}" not in parquet_cols]
print(f"Feature columns: {len(feature_cols)} present, {len(missing_feat)} missing")
if missing_feat:
    print(f"  Missing: {missing_feat}")

# ── Employment group → jbstat OHE column map ─────────────────────────────────
# Maps GROUPS keys (from config_cluster) to the current-wave OHE jbstat column.
# After recode, canonical codes are: 1=Employed, 3=Unemployed, 4=Retired,
# 5=On leave, 7=Student, 8=Inactive.  OHE columns are o_jbstat_<code>.
_JBSTAT_LOOKUP = {
    "Employed":   f"{WAVE}_jbstat_1",
    "Unemployed": f"{WAVE}_jbstat_3",
    "Retired":    f"{WAVE}_jbstat_4",
    "On leave":   f"{WAVE}_jbstat_5",
    "Student":    f"{WAVE}_jbstat_7",
    "Inactive":   f"{WAVE}_jbstat_8",
}
present_jbstat = set(parquet_cols)
GROUP_COL = {
    gname: (
        _JBSTAT_LOOKUP.get(gname)
        if _JBSTAT_LOOKUP.get(gname) in present_jbstat
        else (None if _JBSTAT_LOOKUP.get(gname) is None else "MISSING")
    )
    for gname in GROUPS
}
print(f"\nGroup -> jbstat column (current-wave, mutually exclusive):")
for g, c in GROUP_COL.items():
    print(f"  {g:20s} -> {c}")

# ── Geography units ───────────────────────────────────────────────────────────
if LEVEL_COL is None:
    unit_ids = ["national"]
else:
    # Read only the geography column to get the full list of units
    all_unit_ids = (
        pd.read_parquet(SYNPOP_PARQUET, columns=[LEVEL_COL])[LEVEL_COL]
        .dropna().unique().tolist()
    )
    all_unit_ids.sort()

    # Apply UNIT_FILTER
    if UNIT_FILTER is None:
        unit_ids = all_unit_ids
    elif UNIT_FILTER == "london":
        unit_ids = [u for u in all_unit_ids if u.startswith("E09")]
    elif isinstance(UNIT_FILTER, list):
        unit_ids = [u for u in all_unit_ids if u in set(UNIT_FILTER)]
    else:
        raise ValueError(f"UNIT_FILTER must be None, 'london', or a list of codes")

print(f"\nClustering at '{CLUSTER_LEVEL}' level: {len(unit_ids)} unit(s)"
      + (f" [filter: {UNIT_FILTER!r}]" if UNIT_FILTER else ""))
print(f"Output CSV: {OUTPUT_CSV}")

# ── Cluster loop — one unit at a time ─────────────────────────────────────────
header_written = False

for unit_id in tqdm(unit_ids, desc=CLUSTER_LEVEL):

    # Load only this unit's rows from parquet
    if LEVEL_COL is None:
        df_unit = pd.read_parquet(SYNPOP_PARQUET)
    else:
        df_unit = pd.read_parquet(
            SYNPOP_PARQUET,
            filters=[(LEVEL_COL, '==', unit_id)]
        )

    if df_unit.empty:
        continue

    unit_rows = []
    assigned  = pd.Series(False, index=df_unit.index)

    for gname, gcfg in GROUPS.items():
        col = GROUP_COL.get(gname)

        if col == "MISSING":
            continue
        if col is not None:
            mask = df_unit[col] == 1.0
        else:
            mask = ~assigned   # catch-all

        group_df = df_unit[mask].copy()
        if group_df.empty:
            continue
        assigned |= mask

        avail_feat = [
            c for c in feature_cols
            if c in group_df.columns and group_df[c].notna().any()
        ]
        if not avail_feat:
            continue

        # 1. Fit normalisation on this group
        coeffs = normalise.fit(group_df, avail_feat)

        # 2. Apply normalisation
        group_norm = normalise.apply(group_df, coeffs)

        # 3. KMeans
        k           = gcfg.get('k', K_DEFAULT)
        n           = len(group_norm)
        effective_k = min(k, max(1, n // MIN_CLUSTER_SIZE))
        labels      = cf.fit_kmeans(group_norm[avail_feat].values, effective_k)

        # 4. DNA rows from raw values
        group_df = group_df.copy()
        group_df['_tribe_sub'] = labels

        for sub_id in sorted(group_df['_tribe_sub'].unique()):
            sub_df      = group_df[group_df['_tribe_sub'] == sub_id]
            tribe_label = f"{gname} {sub_id + 1}" if effective_k > 1 else gname
            row = cf.build_dna_row(
                tribe_label, sub_df, WAVE,
                SUMMARY_VARS, VARIABLE_MAP, CATEGORICAL_VARS, CATEGORY_MAPS,
            )
            row['unit_id']       = unit_id
            row['cluster_level'] = CLUSTER_LEVEL
            row['group']         = gname
            unit_rows.append(row)

    # Append this unit's rows to CSV
    if unit_rows:
        unit_df = pd.DataFrame(unit_rows)
        unit_df.to_csv(
            OUTPUT_CSV,
            mode='a',
            header=not header_written,
            index=False,
        )
        header_written = True

    del df_unit, unit_rows
    gc.collect()

print(f"\nDone. Results written to {OUTPUT_CSV}")
if OUTPUT_CSV.exists():
    result = pd.read_csv(OUTPUT_CSV)
    print(f"  {len(result)} tribe rows across {result['unit_id'].nunique()} units")
    print(result[['unit_id', 'cluster_level', 'group', 'tribe_label', 'size']].head(12).to_string(index=False))


Removed existing LA_london_clusters.csv
Feature columns: 12 present, 0 missing

Group -> jbstat column (current-wave, mutually exclusive):
  Employed             -> o_jbstat_1
  Retired              -> o_jbstat_4
  Unemployed           -> o_jbstat_3
  Student              -> o_jbstat_7
  On leave             -> o_jbstat_5
  Inactive             -> o_jbstat_8

Clustering at 'LA' level: 33 unit(s) [filter: 'london']
Output CSV: ../data/6_cluster/LA_london_clusters.csv


LA: 100%|██████████| 33/33 [01:54<00:00,  3.47s/it]


Done. Results written to ../data/6_cluster/LA_london_clusters.csv
  462 tribe rows across 33 units
  unit_id cluster_level      group  tribe_label  size
E09000001            LA   Employed   Employed 1    59
E09000001            LA   Employed   Employed 2  2766
E09000001            LA   Employed   Employed 3  2991
E09000001            LA    Retired    Retired 1   370
E09000001            LA    Retired    Retired 2   876
E09000001            LA    Retired    Retired 3   767
E09000001            LA Unemployed Unemployed 1   293
E09000001            LA Unemployed Unemployed 2   172
E09000001            LA    Student    Student 1    93
E09000001            LA    Student    Student 2   241
E09000001            LA   On leave   On leave 1    71
E09000001            LA   On leave   On leave 2   154
